In [0]:
#1 Load cinema_bookings.csv into a Spark DataFrame with column names and inferred data types.
path="/Workspace/Users/azuser7410_mml.local@karthikirisoutlook.onmicrosoft.com/cinema_bookings.csv"
csv_df=spark.read.csv("file:" + path,
                  header=True,
                  inferSchema=True)
csv_df.show(5)

In [0]:
#2. Load cinema_master.json into a second DataFrame. Display both DataFrames.
json_df=spark.read.json("/Workspace/Users/azuser7410_mml.local@karthikirisoutlook.onmicrosoft.com/cinema_master.json")
json_df.show(5)

In [0]:
#3. Print the schema of both DataFrames. Identify the nested field and array field in the JSON.
csv_df.printSchema()
json_df.printSchema()

In [0]:
#4. Display the first 10 booking records and count the total number of CSV rows.
csv_df.show(10)
csv_df.count()


In [0]:
#5 Select only booking_id, movie_title, tickets and booking_status.
csv_df.select("booking_id","movie_title","tickets","booking_status")

In [0]:
#6. Filter bookings where booking_status is Confirmed.
from pyspark.sql.functions import col
csv_df.filter(col("booking_status")=="Confirmed").show()

In [0]:
#7. Find bookings with more than 3 tickets and a ticket price above 200.
csv_df.filter((col("tickets")>3) & (col("price_per_ticket")>200)).show()


In [0]:
#8. Count missing values in discount_pct and payment_method.
print("Missing value in discount_pct: ", csv_df.filter(col("discount_pct").isNull()).count()) 
print("Missing values in payement_method: ",csv_df.filter(col("payment_method").isNull()).count())

In [0]:
#10. Add gross_amount = tickets multiplied by price_per_ticket.
csv_df=csv_df.withColumn("gross_amount", col("price_per_ticket") * col("tickets"))
csv_df.show()

In [0]:
#11. Add net_amount after applying discount_pct. Keep the amount as a numeric column.
from pyspark.sql.functions import col, round
csv_df=csv_df.withColumn("net_amount", round(col("gross_amount") * (1 - col("discount_pct")/100),2))
csv_df.show()


In [0]:
#12. Add a booking_size column: Large for 4 or more tickets, otherwise Regular.
from pyspark.sql.functions import when
csv_df=csv_df.withColumn("booking_size", when(col("tickets")>=4,"Large").when(col("tickets")<4,"Regular"))
csv_df.show()

In [0]:
csv_df.printSchema()

In [0]:
#13 Sort bookings by net_amount from highest to lowest and show the top 5.
csv_df.orderBy(col("net_amount").desc()).show(5)

In [0]:
#14 Count bookings for each booking_status.
from pyspark.sql.functions import count
cnt=(csv_df.groupBy("booking_status").agg(count("*").alias("count")))
cnt.show()


In [0]:
#15 For Confirmed bookings only, calculate total tickets and net revenue for each movie_title.
from pyspark.sql.functions import sum
cnt=(csv_df.filter(col("booking_status")=="Confirmed").groupBy("movie_title").agg(sum("net_amount").alias("net_revenue"),sum("tickets").alias("total_tickets")))
cnt.show()

In [0]:
#16 Find the top 3 cinema_id values by Confirmed net revenue.
cnt = csv_df.filter(
    col("booking_status") == "Confirmed"
).groupBy("cinema_id").agg(
    sum("net_amount").alias("net_revenue")
)

cnt = cnt.orderBy(col("net_revenue").desc()).limit(3)

cnt.show()

In [0]:
#17 Left join the bookings DataFrame with the cinema master on cinema_id.
joined_df = csv_df.join(
    json_df,
    on="cinema_id",
    how="left"
)

joined_df.show(5)

In [0]:
#18. From the joined DataFrame, show cinema_name, location.city, movie_title and booking_status.
joined_df.select("cinema_name", "location", "movie_title", "booking_status").show()


In [0]:
#19. Calculate Confirmed net revenue by city. Identify bookings with no matching cinema master record.

csv_df=(joined_df.filter(col("booking_status")=="Confirmed").groupBy("location").agg(sum("net_amount").alias("net_revenue")))
csv_df.show()
df2=csv_df.filter(col("location").isNull())
df2.show()

In [0]:
#20 Use the nested JSON field location.zone to display each cinema and its zone.
json_df.select(
    col("cinema_name"),
    col("location.zone").alias("zone")
).show()

In [0]:
#21. Explode the facilities array to produce one row per cinema and facility.
from pyspark.sql.functions import explode
json_df.select(
    col("cinema_name"),
    explode(col("facilities")).alias("facilities")
).show()

In [0]:
# 22. Convert booking_date and show_date into date columns and calculate days_between_booking_and_show.

from pyspark.sql.functions import col, to_date, datediff

csv_df = csv_df.withColumn(
    "booking_date", to_date(col("booking_date"))
).withColumn(
    "show_date", to_date(col("show_date"))
).withColumn(
    "days_between_booking_and_show",
    datediff(col("show_date"), col("booking_date"))
)

csv_df.show(5)

In [0]:
#23 Identify repeated booking_id values and remove duplicate bookings while retaining one row per booking_id.

csv_df.groupBy("booking_id").agg(
    count("*").alias("booking_count")
).filter(
    col("booking_count") > 1
)
csv_df=csv_df.dropDuplicates(["booking_id"])
csv_df.show()

In [0]:
#24 
from pyspark.sql.functions import col, sum, rank
from pyspark.sql.window import Window
movie_revenue = joined_df.filter(
    col("booking_status") == "Confirmed"
).groupBy("movie_title").agg(
    sum("net_amount").alias("confirmed_net_revenue")
)

window_spec = Window.orderBy(col("confirmed_net_revenue").desc())

movie_ranked = movie_revenue.withColumn(
    "movie_rank",
    rank().over(window_spec)
)

movie_ranked.show()

In [0]:
#25. Save the cleaned, deduplicated, joined DataFrame as a Delta table and query five rows with SQL
joined_df.write.format("delta").mode("overwrite").saveAsTable("cinema_bookings_delta")
spark.sql("""
    SELECT *
    FROM cinema_bookings_delta
    LIMIT 5
""").show()

Final challenge (optional)
Build a clean cinema performance report: deduplicate bookings, fill missing values, calculate discounted
revenue for Confirmed bookings, join cinema details, aggregate by city and movie, and save the result as
a Delta table.

In [0]:
cinema_df=json_df

In [0]:
from pyspark.sql.functions import col, sum, count, round
# 1.Remove duplicate bookings
clean_df = csv_df.dropDuplicates(["booking_id"])

# 2.Fill missing values
clean_df = clean_df.fillna({
    "discount_pct": 0,
    "movie_title": "Unknown"
})

# 3.Calculate gross amount
clean_df = clean_df.withColumn(
    "gross_amount",
    col("tickets") * col("price_per_ticket")
)

#4.Calculate discounted net amount
clean_df = clean_df.withColumn(
    "net_amount",
    round(
        col("gross_amount") * (1 - col("discount_pct") / 100),
        2
    )
)

#5.Filter Confirmed bookings
confirmed_df = clean_df.filter(
    col("booking_status") == "Confirmed"
)

#6. LEFT JOIN bookings with cinema master
joined_df = confirmed_df.join(
    cinema_df,
    on="cinema_id",
    how="left"
)

#7. Fill missing city values
joined_df = joined_df.fillna({
    "location": "Unknown"
})

#8. Aggregate by city and movie
report_df = joined_df.groupBy(
    "location",
    "movie_title"
).agg(
    count("booking_id").alias("total_bookings"),
    round(sum("net_amount"), 2).alias("total_net_revenue")
)

#9. Display final report
report_df.orderBy(
    col("total_net_revenue").desc()
).show()

# 10. Save as Delta table
report_df.write.format("delta").mode("overwrite").saveAsTable(
    "cinema_performance_report"
)

# 11. Query Delta table using SQL
spark.sql("""
    SELECT *
    FROM cinema_performance_report
    ORDER BY total_net_revenue DESC
""").show()